# 02 — Préparation

1. Construction de la cible RUL et choix du plafond (120, 125, 130).
2. Conditions de vol (FD002/FD004) par k-moyennes et normalisation **par condition**, apprise sur l'entraînement seulement.
3. Variables mobiles pour les modèles classiques.
4. Fenêtres de séquences pour PyTorch.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
SUBSET = "FD001"   # puis "FD004"


In [ ]:
from src.data import load_subset, add_rul, prepare
from src.features import rolling_features, feature_columns, make_sequences, last_sequences

raw_train, _, _ = load_subset(SUBSET)
rul = add_rul(raw_train, cap=None)
fig, ax = plt.subplots(figsize=(7, 4))
for cap in (None, 120, 125, 130):
    g = add_rul(raw_train, cap)
    g = g[g.unit == 1]
    ax.plot(g.cycle, g.RUL, label=f"plafond {cap}" if cap else "linéaire")
ax.set(xlabel="cycle", ylabel="RUL cible", title="Moteur 1 : cible RUL selon le plafond"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "rul_plafond.png", dpi=150); plt.show()

Le choix du plafond est testé dans le notebook 03 (effet sur la RMSE de validation). On part de 125.

In [ ]:
CAP = 125
train, test, rul_test, sensors = prepare(SUBSET, cap=CAP)
print(len(sensors), "capteurs retenus :", sensors)
print("conditions de vol :", sorted(train.condition.unique()))
train[sensors].describe().loc[["mean", "std"]].round(2)

In [ ]:
WINDOW_FEAT = 10
train_f = rolling_features(train, sensors, WINDOW_FEAT)
cols = feature_columns(sensors)
print(len(cols), "variables pour les modèles classiques")
train_f[["unit", "cycle", "RUL"] + cols[:4]].head()

In [ ]:
WINDOW_SEQ = 30
X, y, groups = make_sequences(train, sensors, WINDOW_SEQ)
X_test, test_units = last_sequences(test, sensors, WINDOW_SEQ)
print("séquences d'entraînement :", X.shape, " test :", X_test.shape)
print("moteurs de test plus courts que la fenêtre :", int((test.groupby('unit').size() < WINDOW_SEQ).sum()))

## À retenir pour le rapport

- La normalisation est apprise sur l'entraînement uniquement (pas de fuite vers le test).
- Les moteurs de test plus courts que la fenêtre sont complétés en répétant leur premier cycle.